In [67]:
# Import
import logs

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mysql.connector
from mysql.connector import errorcode
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")

## Magasin

Définition d'une classe Magasin qui servira à interagir avec la base de donnée SQL

In [80]:
class Magasin:
    def __init__(self, host=logs.host, user=logs.user, port=logs.port,
                 password=logs.password, database=logs.database,
                 orders=None, order_payment=None, order_reviews=None,
                 customers=None, products=None, sellers=None,
                 order_items=None, geolocation=None, stock=None):

        self.host = host
        self.user = user
        self.port = port
        self.password = password
        self.database_name = database

        # Connexion à la base de données
        try:
            self.connection = mysql.connector.connect(
                host=self.host,
                user=self.user,
                port=self.port,
                password=self.password,
                database=self.database_name
            )
            print("Connexion à la base réussie.")
        except mysql.connector.Error as err:
            if err.errno == errorcode.ER_ACCESS_DENIED_ERROR:
                print("Erreur : identifiants incorrects.")
            elif err.errno == errorcode.ER_BAD_DB_ERROR:
                print("Erreur : base de données inexistante.")
            else:
                print(f"Erreur MySQL : {err}")
            raise

        # Chargement des tables
        self.orders = orders if orders is not None else pd.read_sql(
            "SELECT * FROM orders", self.connection)
        self.order_payments = order_payment if order_payment is not None else pd.read_sql(
            "SELECT * FROM order_payments", self.connection)
        self.order_reviews = order_reviews if order_reviews is not None else pd.read_sql(
            "SELECT * FROM order_reviews", self.connection)
        self.customers = customers if customers is not None else pd.read_sql(
            "SELECT * FROM customers", self.connection)
        self.products = products if products is not None else pd.read_sql(
            "SELECT * FROM products", self.connection)
        self.sellers = sellers if sellers is not None else pd.read_sql(
            "SELECT * FROM sellers", self.connection)
        self.order_items = order_items if order_items is not None else pd.read_sql(
            "SELECT * FROM order_items", self.connection)
        self.stock = stock if stock is not None else pd.read_sql(
            "SELECT * FROM stock", self.connection)
        self.geolocation = geolocation if geolocation is not None else pd.read_sql(
            "SELECT * FROM geolocation", self.connection)

        self.database = None

    #Vue globale

    def describe_database(self):
        """
        Fusionne les principales tables autour des commandes
        et renvoie un describe() numérique.
        """
        df = self.orders.copy()

        df = df.merge(self.customers, on="customer_id", how="left")
        df = df.merge(self.order_items, on="order_id", how="left")
        df = df.merge(self.products, on="product_id", how="left")
        df = df.merge(self.sellers, on="seller_id", how="left")
        df = df.merge(self.order_payments, on="order_id", how="left")
        df = df.merge(self.order_reviews, on="order_id", how="left")
        df = df.merge(self.stock, on=["product_id", "seller_id"], how="left")

        self.database = df
        return df.describe()

    #Validation interne

    @staticmethod
    def _check_non_empty_string(value, field_name):
        if not isinstance(value, str) or value.strip() == "":
            raise ValueError(f"{field_name} ne doit pas être vide.")

    @staticmethod
    def _check_positive_number(value, field_name):
        try:
            v = float(value)
        except (TypeError, ValueError):
            raise ValueError(f"{field_name} doit être un nombre.")
        if v <= 0:
            raise ValueError(f"{field_name} doit être strictement positif.")
        return v

    @staticmethod
    def _check_year(value, field_name):
        try:
            year = int(value)
        except (TypeError, ValueError):
            raise ValueError(f"{field_name} doit être un entier (année).")
        current_year = datetime.now().year
        if year < 1980 or year > current_year + 1:
            raise ValueError(f"{field_name} doit être entre 1980 et {current_year + 1}.")
        return year

    #Opérations produits

    def add_product(self, product_name, product_category, product_platform,
                    product_esrb_rating, product_release_year,
                    product_price, product_weight_g, product_description):
        """Ajoute un produit au catalogue avec tests de base."""

        # Validations
        self._check_non_empty_string(product_name, "Nom du produit")
        self._check_non_empty_string(product_category, "Catégorie du produit")
        self._check_non_empty_string(product_platform, "Plateforme")
        self._check_non_empty_string(product_esrb_rating, "Classification ESRB")

        product_release_year = self._check_year(product_release_year, "Année de sortie")
        product_price = self._check_positive_number(product_price, "Prix")
        product_weight_g = self._check_positive_number(product_weight_g, "Poids (g)")

        cursor = self.connection.cursor()
        query = """
            INSERT INTO products
            (product_id, product_name, product_category, product_platform,
             product_esrb_rating, product_release_year, product_price,
             product_weight_g, product_description)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """

        product_id = "PROD_" + "{:05d}".format(self.products.shape[0] + 1)

        try:
            cursor.execute(query, (
                product_id, product_name.strip(), product_category.strip(),
                product_platform.strip(), product_esrb_rating.strip(),
                int(product_release_year), float(product_price),
                int(product_weight_g), product_description
            ))
            self.connection.commit()
            print(f"Produit {product_id} ajouté avec succès.")
        except mysql.connector.Error as err:
            print(f"Erreur MySQL lors de l'ajout du produit : {err}")
            self.connection.rollback()
        finally:
            cursor.close()

        self.products = pd.read_sql("SELECT * FROM products", self.connection)

    def del_product(self, product_id):
        """Supprime un produit s'il existe."""

        self._check_non_empty_string(product_id, "product_id")

        if product_id not in set(self.products["product_id"]):
            print("Aucun produit avec cet identifiant.")
            return

        cursor = self.connection.cursor()
        query = "DELETE FROM products WHERE product_id = %s"
        try:
            cursor.execute(query, (product_id,))
            self.connection.commit()
            print(f"Produit {product_id} supprimé.")
        except mysql.connector.Error as err:
            print(f"Erreur MySQL lors de la suppression : {err}")
            self.connection.rollback()
        finally:
            cursor.close()

        self.products = pd.read_sql("SELECT * FROM products", self.connection)

    def modify_products(self, product_id, new_product_name=None,
                        new_product_category=None, new_product_platform=None,
                        new_product_esrb_rating=None,
                        new_product_release_year=None,
                        new_product_price=None,
                        new_product_weight_g=None,
                        new_product_description=None):
        """Modifie un produit ; seuls les champs non None sont mis à jour."""

        self._check_non_empty_string(product_id, "product_id")

        if product_id not in set(self.products["product_id"]):
            print("Aucun produit avec cet identifiant.")
            return

        # Préparation des champs à mettre à jour
        fields = []
        values = []

        if new_product_name is not None:
            self._check_non_empty_string(new_product_name, "Nom du produit")
            fields.append("product_name = %s")
            values.append(new_product_name.strip())

        if new_product_category is not None:
            self._check_non_empty_string(new_product_category, "Catégorie")
            fields.append("product_category = %s")
            values.append(new_product_category.strip())

        if new_product_platform is not None:
            self._check_non_empty_string(new_product_platform, "Plateforme")
            fields.append("product_platform = %s")
            values.append(new_product_platform.strip())

        if new_product_esrb_rating is not None:
            self._check_non_empty_string(new_product_esrb_rating, "ESRB")
            fields.append("product_esrb_rating = %s")
            values.append(new_product_esrb_rating.strip())

        if new_product_release_year is not None:
            year = self._check_year(new_product_release_year, "Année de sortie")
            fields.append("product_release_year = %s")
            values.append(int(year))

        if new_product_price is not None:
            price = self._check_positive_number(new_product_price, "Prix")
            fields.append("product_price = %s")
            values.append(float(price))

        if new_product_weight_g is not None:
            weight = self._check_positive_number(new_product_weight_g, "Poids (g)")
            fields.append("product_weight_g = %s")
            values.append(int(weight))

        if new_product_description is not None:
            fields.append("product_description = %s")
            values.append(new_product_description)

        if not fields:
            print("Aucune modification demandée.")
            return

        set_clause = ", ".join(fields)
        values.append(product_id)

        cursor = self.connection.cursor()
        query = f"UPDATE products SET {set_clause} WHERE product_id = %s"

        try:
            cursor.execute(query, tuple(values))
            self.connection.commit()
            print(f"Produit {product_id} mis à jour.")
        except mysql.connector.Error as err:
            print(f"Erreur MySQL lors de la mise à jour : {err}")
            self.connection.rollback()
        finally:
            cursor.close()

        self.products = pd.read_sql("SELECT * FROM products", self.connection)

    #Filtres produits

    def filter_products_id(self, product_id):
        return self.products[self.products["product_id"] == product_id]

    def filter_products_name(self, product_name):
        return self.products[self.products["product_name"].str.contains(product_name, case=False)]

    def filter_products_price(self, min_price, max_price):
        min_p = float(min_price)
        max_p = float(max_price)
        return self.products[
            (self.products["product_price"] >= min_p)
            & (self.products["product_price"] <= max_p)
        ]

    def filter_products_category(self, product_category):
        return self.products[self.products["product_category"] == product_category]

    #Connexion

    def close_connection(self):
        if self.connection.is_connected():
            self.connection.close()
            print("Connexion fermée.")

magasin = Magasin()

Connexion à la base réussie.


### Création d'un DataFrame de la base donnée

Uniquement pour avoir un aperçu temporaire

In [81]:
# Description de la Database
magasin.describe_database()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,zip_code_prefix_x,registration_date,last_purchase_date,total_orders,total_spent,order_item_id,...,payment_value,payment_approved_at,review_score,review_creation_date,quantity_in_stock,quantity_reserved,quantity_available,min_stock_level,reorder_point,last_updated
count,17,15,15,14,17.000000,17,16,17.000000,17.000000,17.000000,...,17.000000,15,12.000000,12,16.00000,16.000000,16.00000,16.000000,16.000000,16
mean,2023-10-29 15:01:28.235294208,2023-10-27 04:40:20.000000256,2023-10-27 22:05:20,2023-10-25 04:27:29.999999744,53647.588235,2023-01-25 18:25:52.941176576,2023-12-22 06:37:11.250000128,5.470588,335.671765,1.294118,...,99.334706,2023-10-27 04:40:20.000000256,4.416667,2023-11-14 13:03:20,19.00000,3.250000,15.75000,7.000000,13.750000,2024-01-09 00:42:06
min,2023-07-12 16:00:00,2023-07-12 16:20:00,2023-07-13 09:10:00,2023-07-14 18:30:00,13001.000000,2022-09-21 16:50:00,2023-10-03 14:00:00,1.000000,39.990000,1.000000,...,49.990000,2023-07-12 16:20:00,2.000000,2023-08-28 16:30:00,10.00000,1.000000,9.00000,4.000000,8.000000,2024-01-06 09:20:00
25%,2023-10-01 13:25:00,2023-09-25 15:55:00,2023-09-26 09:05:00,2023-09-22 12:52:30,44000.000000,2022-11-08 09:15:00,2023-12-15 12:50:00,3.000000,189.900000,1.000000,...,74.980000,2023-09-25 15:55:00,4.000000,2023-10-10 18:45:00,14.75000,2.000000,11.00000,5.750000,11.500000,2024-01-08 04:55:00
50%,2023-11-10 17:30:00,2023-11-10 17:50:00,2023-11-11 09:20:00,2023-11-13 14:10:00,59000.000000,2023-01-15 10:30:00,2024-01-01 04:55:00,6.000000,349.900000,1.000000,...,99.980000,2023-11-10 17:50:00,5.000000,2023-11-20 14:40:00,18.00000,3.000000,16.00000,6.500000,13.000000,2024-01-09 00:49:00
75%,2023-12-03 10:10:00,2023-12-02 12:45:00,2023-12-03 09:25:00,2023-12-04 18:10:00,69002.000000,2023-03-02 14:10:00,2024-01-06 22:47:30,7.000000,429.400000,2.000000,...,114.980000,2023-12-02 12:45:00,5.000000,2023-12-06 22:20:00,22.00000,4.000000,18.00000,8.000000,16.000000,2024-01-10 15:02:28
max,2024-01-05 18:15:00,2024-01-05 18:40:00,2024-01-06 09:30:00,2024-01-08 19:00:00,75001.000000,2023-09-10 15:25:00,2024-01-10 18:10:00,9.000000,599.800000,2.000000,...,149.980000,2024-01-05 18:40:00,5.000000,2024-01-12 19:00:00,30.00000,5.000000,25.00000,10.000000,20.000000,2024-01-10 15:07:10
std,NaN,NaN,NaN,NaN,19170.443908,NaN,NaN,2.695312,188.056867,0.469668,...,29.197305,NaN,0.996205,NaN,5.80804,1.238278,4.86484,1.932184,3.623994,NaN


### Filtre

Utilisation du nom, prix et catégorie pour filtrer

In [82]:
# Filtre par nom
magasin.filter_products_name("Elden Ring")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description,created_at
0,PROD_00001,Elden Ring,Action RPG,PS5,16+,2022,59.99,150,Open world dark fantasy RPG,2022-02-25


In [83]:
# Filtre par prix
magasin.filter_products_price(0, 50)

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description,created_at
3,PROD_00004,Mario Kart 8 Deluxe,Racing,Switch,3+,2017,49.99,120,Arcade kart racing game,2017-04-28
6,PROD_00007,Animal Crossing: NH,Simulation,Switch,3+,2020,49.99,110,Life simulation game,2020-03-20
9,PROD_00010,Just Dance 2024,Music,Switch,3+,2023,49.99,120,Dance rhythm game,2023-10-24


In [84]:
# Filtre par catégorie
magasin.filter_products_category("Action RPG")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description,created_at
0,PROD_00001,Elden Ring,Action RPG,PS5,16+,2022,59.99,150,Open world dark fantasy RPG,2022-02-25
2,PROD_00003,The Legend of Zelda: TOTK,Action RPG,Switch,12+,2023,69.99,110,Adventure in the kingdom of Hyrule,2023-05-12
8,PROD_00009,Hogwarts Legacy,Action RPG,PS5,16+,2023,69.99,150,Wizarding World adventure,2023-02-10


### Ajout et suppression

Fonction d'ajout et suppression d'un produit

In [85]:
# Ajoute un produit
magasin.add_product("Pokemon Platine", "RPG Aventure", "DS", "3+", "2007", "24.99", "254", "Game : Pokemon Platine")

magasin.filter_products_name("Pokemon Platine")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description,created_at
10,PROD_00011,Pokemon Platine,RPG Aventure,DS,3+,2007,24.99,254,Game : Pokemon Platine,2026-02-05 15:44:25


In [86]:
# Supprime un produit
magasin.del_product("PROD_00011")

magasin.filter_products_name("Pokemon Platine")

,product_id,product_name,product_category,product_platform,product_esrb_rating,product_release_year,product_price,product_weight_g,product_description,created_at


### Interface Console

In [89]:
def menu_console():
    magasin = Magasin()

    while True:
        print("\n===== MENU MAGASIN =====")
        print("1 - Ajouter un produit")
        print("2 - Modifier un produit")
        print("3 - Supprimer un produit")
        print("4 - Rechercher un produit (id)")
        print("5 - Rechercher un produit (nom)")
        print("6 - Rechercher par intervalle de prix")
        print("7 - Rechercher par catégorie")
        print("8 - Afficher un aperçu de la base (describe)")
        print("0 - Quitter")
        choix = input("Votre choix : ").strip()

        if choix == "1":
            print("\n--- Ajout d'un produit ---")
            try:
                name = input("Nom du produit : ")
                cat = input("Catégorie : ")
                plat = input("Plateforme (PS5, Switch, etc.) : ")
                esrb = input("Classification ESRB (3+, 12+, 16+, 18+) : ")
                year = input("Année de sortie (ex: 2023) : ")
                price = input("Prix (€) : ")
                weight = input("Poids (g) : ")
                desc = input("Description (optionnelle) : ")

                magasin.add_product(name, cat, plat, esrb, year, price, weight, desc)
            except ValueError as e:
                print(f"Erreur de validation : {e}")

        elif choix == "2":
            print("\n--- Modification d'un produit ---")
            pid = input("ID du produit à modifier (ex: PROD_00001) : ").strip()
            print("Laisse une entrée vide pour ne pas modifier le champ.")
            name = input("Nouveau nom (ou vide) : ")
            cat = input("Nouvelle catégorie (ou vide) : ")
            plat = input("Nouvelle plateforme (ou vide) : ")
            esrb = input("Nouvelle classification ESRB (ou vide) : ")
            year = input("Nouvelle année de sortie (ou vide) : ")
            price = input("Nouveau prix (ou vide) : ")
            weight = input("Nouveau poids (g) (ou vide) : ")
            desc = input("Nouvelle description (ou vide) : ")

            kwargs = {}
            if name.strip():
                kwargs["new_product_name"] = name
            if cat.strip():
                kwargs["new_product_category"] = cat
            if plat.strip():
                kwargs["new_product_platform"] = plat
            if esrb.strip():
                kwargs["new_product_esrb_rating"] = esrb
            if year.strip():
                kwargs["new_product_release_year"] = year
            if price.strip():
                kwargs["new_product_price"] = price
            if weight.strip():
                kwargs["new_product_weight_g"] = weight
            if desc.strip():
                kwargs["new_product_description"] = desc

            try:
                magasin.modify_products(pid, **kwargs)
            except ValueError as e:
                print(f"Erreur de validation : {e}")

        elif choix == "3":
            print("\n--- Suppression d'un produit ---")
            pid = input("ID du produit à supprimer : ").strip()
            try:
                magasin.del_product(pid)
            except ValueError as e:
                print(f"Erreur : {e}")

        elif choix == "4":
            pid = input("ID du produit : ").strip()
            print(magasin.filter_products_id(pid))

        elif choix == "5":
            name = input("Nom ou fragment de nom : ").strip()
            print(magasin.filter_products_name(name))

        elif choix == "6":
            min_p = input("Prix minimum : ")
            max_p = input("Prix maximum : ")
            try:
                print(magasin.filter_products_price(min_p, max_p))
            except ValueError:
                print("Veuillez saisir des nombres valides pour les prix.")

        elif choix == "7":
            cat = input("Catégorie : ").strip()
            print(magasin.filter_products_category(cat))

        elif choix == "8":
            print("\n--- Aperçu statistique ---")
            print(magasin.describe_database())

        elif choix == "0":
            magasin.close_connection()
            print("Au revoir.")
            break

        else:
            print("Choix invalide, veuillez réessayer.")


if __name__ == "__main__":
    menu_console()

Connexion fermée.
Au revoir.


### Fermer la connexion à la base de donnée

In [61]:
# Close connection
magasin.close_connection()